In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2M, preprocess_input as efficientnet_preprocessing
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.losses import CategoricalCrossentropy



In [ ]:
# Kaggle setup
!pip install -q kaggle
from google.colab import files
files.upload()
! mkdir ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
import kagglehub
path = kagglehub.dataset_download("aneesh10/cricket-shot-dataset")
data_dir = os.path.join(path, "data")



In [ ]:
# Get class names
class_names = os.listdir(data_dir)

# Creating dataframe
image_paths, class_labels = [], []
for class_name in class_names:
    paths = glob(os.path.join(data_dir, class_name) + '/*.png')
    image_paths.extend(paths)
    class_labels.extend([class_name] * len(paths))
df = pd.DataFrame({'image_path': image_paths, 'label': class_labels})

# Splitting data
train_df, temp_df = train_test_split(df, stratify=df[['label']], test_size=0.3, random_state=40)
val_df, test_df = train_test_split(temp_df, stratify=temp_df[['label']], test_size=0.5, random_state=40)

# Data generators
train_gen = ImageDataGenerator(preprocessing_function=efficientnet_preprocessing, horizontal_flip=True, vertical_flip=True)
val_test_gen = ImageDataGenerator(preprocessing_function=efficientnet_preprocessing)

train_generator = train_gen.flow_from_dataframe(train_df, x_col="image_path", y_col="label", target_size=(256, 256), batch_size=32, class_mode='categorical', shuffle=True)
valid_generator = val_test_gen.flow_from_dataframe(val_df, x_col="image_path", y_col="label", target_size=(256, 256), batch_size=32, class_mode='categorical', shuffle=True)
test_generator = val_test_gen.flow_from_dataframe(test_df, x_col="image_path", y_col="label", target_size=(256, 256), batch_size=1, class_mode='categorical', shuffle=False)



In [ ]:
# Model setup
base_model = EfficientNetV2M(input_shape=(256, 256, 3), include_top=False, weights='imagenet')
base_model.trainable = False
x = Flatten()(base_model.output)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.1)(x)
x = Dense(32, activation='relu')(x)
out = Dense(len(class_names), activation='softmax')(x)
model = Model(base_model.input, out)

# Compile model
model.compile(optimizer=Adam(learning_rate=0.0003), loss=CategoricalCrossentropy(), metrics=['accuracy'])

# Training
callbacks = [ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1), EarlyStopping(monitor='val_loss', patience=5, start_from_epoch=10)]
history = model.fit(train_generator, epochs=10, validation_data=valid_generator, callbacks=callbacks, verbose=1)



In [ ]:
# Plot results
# Set a clean seaborn style
sns.set_style("whitegrid")

# Create figure
plt.figure(figsize=(12, 5))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss', color='red')
plt.plot(history.history['val_loss'], label='Validation Loss', color='green')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training vs Validation Loss')

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='red')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='green')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training vs Validation Accuracy')

plt.tight_layout()
plt.show()